# Destination-port sensitivity on CICIDS2017

A small experiment inspired by Altidor and Talhi (CIoT 2024), not a reproduction of their genetic algorithm. Saved outputs are from the supplied executed notebook. Code and outputs are preserved.

## 1. Locate the two source CSV files
Place the PortScan and DDoS CSV files beside this notebook, or in `data` or `MachineLearningCVE`. Colab users can use the configured Drive folder.


In [45]:
import os, glob

DATA_DIR = "/content/drive/MyDrive/CICIDS2017"   # Colab: folder in Google Drive with the two CSV files

try:
    from google.colab import drive
    drive.mount("/content/drive")
    search_dirs = [DATA_DIR, "/content"]
except ImportError:                               # running on your own computer
    search_dirs = [".", "MachineLearningCVE", "data"]

files = [f for d in search_dirs for f in glob.glob(os.path.join(d, "*.csv"))]
portscan_file = [f for f in files if "portscan" in f.lower()]
ddos_file = [f for f in files if "ddos" in f.lower()]
print("PortScan file:", portscan_file)
print("DDoS file:    ", ddos_file)
assert portscan_file and ddos_file, "Files not found. Check the folder name and that both CSV files are inside it."

PortScan file: ['.\\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv']
DDoS file:     ['.\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']


## 2. Load the flows
Combine the two Friday-afternoon files and convert predictors to numeric values. There are 78 original predictors plus the class label.


In [46]:
import numpy as np
import pandas as pd

df = pd.concat([pd.read_csv(portscan_file[0], low_memory=False),
                pd.read_csv(ddos_file[0], low_memory=False)], ignore_index=True)
df.columns = df.columns.str.strip()

feature_cols = [c for c in df.columns if c != "Label"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

print("Rows, columns:", df.shape)
print(df["Label"].value_counts(), "\n")

show = ["Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Flow Bytes/s", "Flow Packets/s", "SYN Flag Count", "Label"]
df.groupby("Label").head(2)[show]

Rows, columns: (512212, 79)
Label
BENIGN      225255
PortScan    158930
DDoS        128027
Name: count, dtype: int64 



,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Bytes/s,Flow Packets/s,SYN Flag Count,Label
0,22,1266342,41,44,7595.104640,67.122468,0,BENIGN
1,22,1319353,41,44,7289.936810,64.425518,0,BENIGN
1463,80,5021059,6,5,421.624203,2.190773,0,PortScan
1467,80,70,1,1,0.000000,28571.428570,0,PortScan
305350,80,1293792,3,7,8991.398927,7.729218,0,DDoS
305351,80,4421382,4,0,5.428167,0.904695,0,DDoS


## 3. Clean the data
Remove nonfinite/missing rows and exact duplicate rows, then remove constant and identical predictor columns. This run retained 435,336 flows and 60 predictors.

**Limitation:** column filtering is performed on the combined dataset before splitting; it is unsupervised but uses held-out predictor values. For a stricter evaluation, fit preprocessing on training data only. Exact row deduplication is performed before column filtering; smaller projections can still contain identical or near-identical predictor rows across splits.


In [47]:
before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.drop_duplicates()
df = df[df["Label"].isin(["BENIGN", "PortScan", "DDoS"])]
print(f"Removed {before - len(df):,} rows. Remaining: {len(df):,}")
print(df["Label"].value_counts(), "\n")

feature_cols = [c for c in df.columns if c != "Label"]
constant = [c for c in feature_cols if df[c].nunique() <= 1]
rest = [c for c in feature_cols if c not in constant]
duplicate = list(df[rest].T.index[df[rest].T.duplicated()])
df = df.drop(columns=constant + duplicate)
print(f"Constant columns removed ({len(constant)}):", constant)
print(f"Duplicate columns removed ({len(duplicate)}):", duplicate)
print("Features left:", df.shape[1] - 1)

Removed 76,876 rows. Remaining: 435,336
Label
BENIGN      216628
DDoS        128014
PortScan     90694
Name: count, dtype: int64 

Constant columns removed (10): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Duplicate columns removed (8): ['SYN Flag Count', 'ECE Flag Count', 'Avg Fwd Segment Size', 'Fwd Header Length.1', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes']
Features left: 60


## 4. Inspect destination-port distributions
DDoS flows almost exclusively use port 80; rounded percentages do not imply every flow uses that port. PortScan covers 1,000 ports.


In [48]:
for label in ["BENIGN", "PortScan", "DDoS"]:
    ports = df.loc[df["Label"] == label, "Destination Port"]
    top = (ports.value_counts(normalize=True).head(5) * 100).round(2)
    print(f"{label}: {ports.nunique():,} different ports. Top 5 ports (% of flows):")
    print(top.to_string(), "\n")

BENIGN: 30,794 different ports. Top 5 ports (% of flows):
Destination Port
53      40.00
443     17.63
80      11.02
8080     0.52
123      0.44 

PortScan: 1,000 different ports. Top 5 ports (% of flows):
Destination Port
80     0.41
22     0.27
21     0.27
139    0.22
445    0.20 

DDoS: 4 different ports. Top 5 ports (% of flows):
Destination Port
80       100.0
64869      0.0
64873      0.0
27636      0.0 



## 5. Sample and split
Sample 40,000 benign, 20,000 PortScan and 20,000 DDoS flows. Use a stratified 70/30 split (56,000 training and 24,000 test flows). This is a class-balanced experiment, not a deployment prevalence estimate.


In [49]:
from sklearn.model_selection import train_test_split

N = min(20000, (df["Label"] == "PortScan").sum(), (df["Label"] == "DDoS").sum())
n_benign = min(2 * N, (df["Label"] == "BENIGN").sum())

sample = pd.concat([
    df[df["Label"] == "BENIGN"].sample(n_benign, random_state=42),
    df[df["Label"] == "PortScan"].sample(N, random_state=42),
    df[df["Label"] == "DDoS"].sample(N, random_state=42),
])
X = sample.drop(columns=["Label"])
y = sample["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("Train:", X_train.shape, "  Test:", X_test.shape)
print(y_train.value_counts())

Train: (56000, 60)   Test: (24000, 60)
Label
BENIGN      28000
DDoS        14000
PortScan    14000
Name: count, dtype: int64


## 6. Define the model and perturbation
Random Forest uses 100 trees and seed 42. Replace test-set ports 80, 443 and 53 with random integers from 1024 through 65535 (perturbation seed 0). Other features and labels remain unchanged. Both benign and attack flows can be modified. This is a tabular sensitivity test, not regenerated traffic or a demonstration of a real attack evasion.


In [50]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

LABELS = ["BENIGN", "PortScan", "DDoS"]
results = []

def train(X_tr, y_tr):
    model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X_tr, y_tr)
    return model

def evaluate(name, model, X_te, y_te):
    pred = model.predict(X_te)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average="macro")
    results.append({"Experiment": name, "Accuracy (%)": round(acc * 100, 2), "Macro F1 (%)": round(f1 * 100, 2)})
    print(f"{name}\n  Accuracy: {acc:.2%}   Macro F1: {f1:.2%}")
    cm = pd.DataFrame(confusion_matrix(y_te, pred, labels=LABELS),
                      index=["true " + l for l in LABELS], columns=["pred " + l for l in LABELS])
    print(cm, "\n")

def obfuscate_ports(X_te, ports=(80, 443, 53), seed=0):
    X_new = X_te.copy()
    mask = X_new["Destination Port"].isin(ports)
    rng = np.random.default_rng(seed)
    X_new.loc[mask, "Destination Port"] = rng.integers(1024, 65536, mask.sum())
    print(f"Changed the port of {mask.sum():,} of {len(X_new):,} test flows")
    return X_new

## 7. Full-feature baseline (60 predictors)


In [51]:
model_A = train(X_train, y_train)
evaluate("A: all features", model_A, X_test, y_test)

A: all features
  Accuracy: 99.95%   Macro F1: 99.95%
               pred BENIGN  pred PortScan  pred DDoS
true BENIGN          11998              1          1
true PortScan            0           6000          0
true DDoS                9              0       5991 



## 8. Perturb ports with the same trained full-feature model
No retraining is performed.


In [52]:
X_test_obf = obfuscate_ports(X_test)
evaluate("B: all features, ports changed in test", model_A, X_test_obf, y_test)

Changed the port of 14,280 of 24,000 test flows
B: all features, ports changed in test
  Accuracy: 99.89%   Macro F1: 99.89%
               pred BENIGN  pred PortScan  pred DDoS
true BENIGN          11999              1          0
true PortScan            0           6000          0
true DDoS               25              0       5975 



## 9. Retrain without destination port (59 predictors)


In [53]:
model_C = train(X_train.drop(columns=["Destination Port"]), y_train)
evaluate("C: port feature removed", model_C, X_test.drop(columns=["Destination Port"]), y_test)

C: port feature removed
  Accuracy: 99.95%   Macro F1: 99.95%
               pred BENIGN  pred PortScan  pred DDoS
true BENIGN          11999              0          1
true PortScan            0           6000          0
true DDoS               10              0       5990 



## 10. Rank training-set feature importance
Impurity-based importance is used for subset construction, not as proof of causal importance. Destination port ranks 21st in this run.


In [54]:
importance = pd.Series(model_A.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print((importance.head(10) * 100).round(2).to_string())
rank = list(importance.index).index("Destination Port") + 1
print(f"\nDestination Port is feature #{rank} of {len(importance)}")

Fwd Packet Length Mean         8.77
Total Length of Fwd Packets    8.23
Fwd Packet Length Max          8.14
Packet Length Mean             6.66
Init_Win_bytes_forward         5.17
Average Packet Size            4.58
Bwd Packets/s                  3.63
Flow IAT Max                   3.50
Flow Duration                  3.48
Bwd Packet Length Std          2.97

Destination Port is feature #21 of 60


## 11. Small subset and port ablation
Use the top 13 training-model features, replacing the 13th with destination port when absent. In this run this produces the 12 highest-ranked features plus the port. This is not the paper's GA-selected subset. D3 retrains on the remaining 12 features.

After perturbation, 2,194 of 6,000 DDoS test flows are predicted benign (36.57%); none are predicted PortScan.


In [55]:
top13 = list(importance.head(13).index)
port_in_top13 = "Destination Port" in top13
if not port_in_top13:
    top13 = top13[:12] + ["Destination Port"]
print("Port naturally in the top 13:", port_in_top13)
print("13 features:", top13, "\n")

model_D = train(X_train[top13], y_train)
evaluate("D1: 13 features", model_D, X_test[top13], y_test)
evaluate("D2: 13 features, ports changed in test", model_D, X_test_obf[top13], y_test)

top12 = [c for c in top13 if c != "Destination Port"]
model_D3 = train(X_train[top12], y_train)
evaluate("D3: same 13 features minus port", model_D3, X_test[top12], y_test)

Port naturally in the top 13: False
13 features: ['Fwd Packet Length Mean', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Packet Length Mean', 'Init_Win_bytes_forward', 'Average Packet Size', 'Bwd Packets/s', 'Flow IAT Max', 'Flow Duration', 'Bwd Packet Length Std', 'Bwd Packet Length Mean', 'Avg Bwd Segment Size', 'Destination Port'] 

D1: 13 features
  Accuracy: 99.96%   Macro F1: 99.96%
               pred BENIGN  pred PortScan  pred DDoS
true BENIGN          11999              1          0
true PortScan            1           5999          0
true DDoS                7              0       5993 

D2: 13 features, ports changed in test
  Accuracy: 90.83%   Macro F1: 89.73%
               pred BENIGN  pred PortScan  pred DDoS
true BENIGN          11999              1          0
true PortScan            5           5995          0
true DDoS             2194              0       3806 

D3: same 13 features minus port
  Accuracy: 99.95%   Macro F1: 99.95%
               pred B

## 12. Main summary
Export the six main experiments to `results.csv`.


In [56]:
summary = pd.DataFrame(results)
summary.to_csv("results.csv", index=False)
summary

,Experiment,Accuracy (%),Macro F1 (%)
0,A: all features,99.95,99.95
1,"B: all features, ports changed in test",99.89,99.89
2,C: port feature removed,99.95,99.95
3,D1: 13 features,99.96,99.96
4,"D2: 13 features, ports changed in test",90.83,89.73
5,D3: same 13 features minus port,99.95,99.95


## 13. Repeat with three seeds
Seeds 42, 1 and 7 change sampling, train/test splitting, model randomness and subset selection together. The perturbation RNG seed remains 0. These are three overlapping samples from the same source data, not independent datasets. “DDoS missed” means predicted as any class other than DDoS.


In [57]:
rows = []
for seed in [42, 1, 7]:
    smp = pd.concat([
        df[df["Label"] == "BENIGN"].sample(n_benign, random_state=seed),
        df[df["Label"] == "PortScan"].sample(N, random_state=seed),
        df[df["Label"] == "DDoS"].sample(N, random_state=seed),
    ])
    Xs, ys = smp.drop(columns=["Label"]), smp["Label"]
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, stratify=ys, random_state=seed)
    m_all = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed).fit(Xtr, ytr)
    imp = pd.Series(m_all.feature_importances_, index=Xs.columns).sort_values(ascending=False)
    t13 = list(imp.head(13).index)
    if "Destination Port" not in t13:
        t13 = t13[:12] + ["Destination Port"]
    m13 = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed).fit(Xtr[t13], ytr)
    Xte_obf = obfuscate_ports(Xte)
    pred_obf = m13.predict(Xte_obf[t13])
    is_ddos = (yte == "DDoS").values
    rows.append({
        "Seed": seed,
        "Port rank (all features)": list(imp.index).index("Destination Port") + 1,
        "D1 accuracy (%)": round(accuracy_score(yte, m13.predict(Xte[t13])) * 100, 2),
        "D2 accuracy (%)": round(accuracy_score(yte, pred_obf) * 100, 2),
        "DDoS missed in D2 (%)": round((pred_obf[is_ddos] != "DDoS").mean() * 100, 1),
    })
pd.DataFrame(rows)

Changed the port of 14,280 of 24,000 test flows
Changed the port of 14,367 of 24,000 test flows
Changed the port of 14,255 of 24,000 test flows


,Seed,Port rank (all features),D1 accuracy (%),D2 accuracy (%),DDoS missed in D2 (%)
0,42,21,99.96,90.83,36.6
1,1,22,99.96,90.92,36.1
2,7,20,99.97,90.68,37.2


## 14. Ten random subsets that always include destination port
Choose 12 distinct non-port features plus destination port. Use the same main train/test split and perturbed test set. These results concern port-containing subsets only; they do not establish a general effect of feature count. “DDoS missed” includes benign and PortScan predictions.


In [58]:
non_port = [c for c in X_train.columns if c != "Destination Port"]
rng_sets = np.random.default_rng(123)
rows = []
for i in range(10):
    feats = list(rng_sets.choice(non_port, 12, replace=False)) + ["Destination Port"]
    m = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42).fit(X_train[feats], y_train)
    pred_obf = m.predict(X_test_obf[feats])
    is_ddos = (y_test == "DDoS").values
    rows.append({
        "Random set": i + 1,
        "Accuracy (%)": round(accuracy_score(y_test, m.predict(X_test[feats])) * 100, 2),
        "Accuracy, ports changed (%)": round(accuracy_score(y_test, pred_obf) * 100, 2),
        "DDoS missed (%)": round((pred_obf[is_ddos] != "DDoS").mean() * 100, 1),
    })
random_sets = pd.DataFrame(rows)
random_sets

,Random set,Accuracy (%),"Accuracy, ports changed (%)",DDoS missed (%)
0,1,99.93,74.97,99.8
1,2,99.96,75.48,97.8
2,3,99.97,83.05,67.7
3,4,99.95,75.28,98.7
4,5,99.94,75.24,98.9
5,6,99.97,75.06,99.6
6,7,99.88,75.19,98.5
7,8,99.93,90.81,36.6
8,9,99.96,90.85,36.6
9,10,99.96,74.95,100.0


## Interpretation and scope

In this run, the full model changes little after port perturbation, whereas selected small subsets containing the port show greater sensitivity. Removing the port yields similar rounded accuracy on the original test split. These observations do not prove robustness to other shifts, generalization to new days/networks, or that the paper's final subset behaves the same way. The original paper used GA-based feature selection and a different evaluation procedure. A temporal/grouped holdout and training-only preprocessing would strengthen a follow-up. No raw dataset files are included.
